# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [13]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [14]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"The total revenue for all orders is ${total_revenue:.2f}.")
print(f"The total number of units sold across all orders is {total_units}.")

The total revenue for all orders is $8520.00.
The total number of units sold across all orders is 783.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [15]:
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).reset_index()
revenue_by_category['share_of_total'] = (revenue_by_category['revenue'] / df['revenue'].sum()) * 100
display(revenue_by_category)

print(f"The table above shows the total revenue for each product category, ordered from the highest revenue-generating category (Food) to the lowest (RainGear). It also displays the percentage each category contributes to the overall total revenue of ${df['revenue'].sum():.2f}.")

,category,revenue,share_of_total
0,Food,4293.0,50.387324
1,Merch,1771.5,20.792254
2,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


The table above shows the total revenue for each product category, ordered from the highest revenue-generating category (Food) to the lowest (RainGear). It also displays the percentage each category contributes to the overall total revenue of $8520.00.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [16]:
vendor_performance = df.groupby('vendor_id').agg(
    average_order_revenue=('revenue', 'mean'),
    order_count=('vendor_id', 'count')
).sort_values(by='average_order_revenue', ascending=False).reset_index()

display(vendor_performance)

highest_avg_vendor = vendor_performance.iloc[0]
print(f"The table above displays each vendor's average order revenue and their total order count. Vendor {highest_avg_vendor['vendor_id']} has the highest average order revenue of ${highest_avg_vendor['average_order_revenue']:.2f}, based on {highest_avg_vendor['order_count']} orders.")

,vendor_id,average_order_revenue,order_count
0,V-01,22.595745,94
1,V-18,21.750000,108
2,V-05,20.580645,93
3,V-10,20.314286,105


The table above displays each vendor's average order revenue and their total order count. Vendor V-01 has the highest average order revenue of $22.60, based on 94 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [17]:
merch_share = revenue_by_category[revenue_by_category['category'] == 'Merch']['share_of_total'].iloc[0]

print(f"The Merch category contributes {merch_share:.1f}% to the total revenue.")

The Merch category contributes 20.8% to the total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [18]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

original_row_count = len(df)
original_total_revenue = df['revenue'].sum()

joined_df = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

current_row_count = len(joined_df)
current_total_revenue = joined_df['revenue'].sum()

print(f"Original row count: {original_row_count}. Current row count after merge: {current_row_count}. The row count has not changed.")
print(f"Original total revenue: ${original_total_revenue:.2f}. Current total revenue after merge: ${current_total_revenue:.2f}. The total revenue has not changed.")

unmatched_vendor_id = joined_df[joined_df['vendor_name'].isnull()]['vendor_id'].unique()[0]
print(f"The unmatched vendor ID is: {unmatched_vendor_id}.")

joined_df['vendor_name'] = joined_df['vendor_name'].fillna('Unmatched Vendor ' + unmatched_vendor_id)

display(joined_df.head())
display(joined_df[joined_df['vendor_id'] == unmatched_vendor_id].head())

Original row count: 400. Current row count after merge: 400. The row count has not changed.
Original total revenue: $8520.00. Current total revenue after merge: $8520.00. The total revenue has not changed.
The unmatched vendor ID is: V-18.


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unmatched Vendor V-18
2,V-18,Drink,3,4.5,13.5,Unmatched Vendor V-18
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unmatched Vendor V-18


,vendor_id,category,qty,price,revenue,vendor_name
1,V-18,RainGear,1,12.0,12.0,Unmatched Vendor V-18
2,V-18,Drink,3,4.5,13.5,Unmatched Vendor V-18
4,V-18,Drink,3,7.5,22.5,Unmatched Vendor V-18
5,V-18,Merch,1,24.0,24.0,Unmatched Vendor V-18
6,V-18,Drink,1,24.0,24.0,Unmatched Vendor V-18


**The unmatched vendor, and what I did about it:**
The vendor_id V-18 was found to be unmatched in the vendor_names DataFrame. To ensure readability and retain all order data, I filled the vendor_name for this vendor with the descriptive string 'Unmatched Vendor V-18'. This allows all orders to be accounted for in the merged DataFrame without losing data or introducing incorrect vendor names. The merge was successful without altering the total row count (400 rows) or the total revenue ($8520.00), confirming the integrity of the data.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [19]:
pivot_table = pd.pivot_table(
    joined_df,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    margins=True,
    fill_value=0
)

display(pivot_table)

print("The table above is a pivot table that shows the total revenue generated by each vendor across different product categories. The 'All' row and column represent the total revenue for each category and vendor, respectively, with the grand total revenue shown in the bottom-right cell.")

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unmatched Vendor V-18,582.0,1018.5,508.5,240.0,2349.0
All,1554.0,4293.0,1771.5,901.5,8520.0


The table above is a pivot table that shows the total revenue generated by each vendor across different product categories. The 'All' row and column represent the total revenue for each category and vendor, respectively, with the grand total revenue shown in the bottom-right cell.


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [20]:
# assert len(df) == 400
# assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
# assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
# assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


In [21]:
by_category = revenue_by_category
joined = joined_df

assert len(df) == 400, f"Expected 400 rows, got {len(df)}"
assert 8000 < df['revenue'].sum() < 9000, f"Expected total revenue between $8000 and $9000, got ${df['revenue'].sum()}"
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01, f"Category sum ({by_category['revenue'].sum()}) does not match total revenue ({df['revenue'].sum()}"
assert len(joined) == len(df), f'The vendor merge changed the row count. Expected {len(df)} rows, got {len(joined)}'
print('All checks passed.')

All checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

**a) What would you tell these vendors to do differently next game?**

To maximize revenue, vendors should closely examine the strategies of 'Unmatched Vendor V-18,' which, despite being unidentified, generated the highest total revenue at $2349.00. This vendor's success, particularly in the 'Food' and 'Drink' categories, indicates a strong market demand that others should emulate. Given that 'Food' alone accounts for 50.39% of the total revenue, all vendors should prioritize and potentially expand their offerings in this category. Furthermore, vendors like 'Hoos Burgers,' which already has a high average order revenue of $22.60, could explore diversifying their menu within high-demand categories to capitalize on their established customer base and pricing power.

**b) Which of your seven answers is the least trustworthy, and why?**

The least trustworthy answer is the analysis concerning the 'Unmatched Vendor V-18.' While the data accurately reflects their total revenue of $2349.00, the lack of a proper vendor name makes any deeper interpretation or targeted advice speculative. I had to assign a generic placeholder, which prevents from understanding their specific business model, customer base, or product details. This introduces an issue: I can quantify their performance, but I cannot ascertain why they performed that way or provide actionable, vendor-specific recommendations beyond broad generalizations. This severely limits the trustworthiness and utility of any specific strategic advice I could offer them.